In [ ]:
5

5

In [ ]:
import pandas as pd

df = pd.read_json('D:/Adrta/HYBRID_DB_PROJECT/imdb_reviews/part-01.json')
print(df.shape)

(1010293, 9)


In [ ]:
df.columns

Index(['review_id', 'reviewer', 'movie', 'rating', 'review_summary',
       'review_date', 'spoiler_tag', 'review_detail', 'helpful'],
      dtype='str')

In [ ]:
df["helpful_votes"] = df["helpful"].apply(lambda x: int(x[0].replace(",", "")))
df["total_votes"] = df["helpful"].apply(lambda x: int(x[1].replace(",", "")))

In [ ]:
df = df[df["helpful_votes"] > 0]

In [ ]:
print("Remaining reviews:", len(df))

Remaining reviews: 646561


In [ ]:
df["document"] = (
    "Movie: " + df["movie"] + "\n" +
    "Rating: " + df["rating"].astype(str) + "\n" +
    "Summary: " + df["review_summary"].fillna("") + "\n" +
    "Review: " + df["review_detail"].fillna("")
)

In [ ]:
vector_df = df[[
    "review_id",
    "document",
    "movie",
    "rating",
    "spoiler_tag",
    "review_date",
    "reviewer"
]]

sql_df = df[[
    "review_id",
    "movie",
    "reviewer",
    "rating",
    "review_date",
    "spoiler_tag",
    "review_summary"
]]

In [ ]:
sql_df = sql_df[sql_df["review_id"].isin(vector_df["review_id"])]

In [ ]:
sql_df.to_csv("sql_data.csv", index=False)

In [ ]:
vector_df.to_json("vector_data.json", orient="records", lines=True)

## Vector DB

In [33]:
import json

open ("vector_data.json", "r").readlines()[:2] 

['{"review_id":"rw5704482","document":"Movie: After Life (2019\\u2013 )\\nRating: 9.0\\nSummary: Very Strong Season 2\\nReview: I enjoyed the first season, but I must say I think season 2 is even stronger. Ricky does a great job as both writer, actor and director and brings out the best in a superb supporting cast. If there was one thing I\'d change, I\'d like to hear him talk about himself less with other people and speak more in the third person, but other than that it\'s pretty hard to fault this funny yet emotional comedy.","movie":"After Life (2019\\u2013 )","rating":9.0,"spoiler_tag":0,"review_date":"3 May 2020","reviewer":"raeldor-96879"}\n',
 '{"review_id":"rw5704483","document":"Movie: The Valhalla Murders (2019\\u2013 )\\nRating: 6.0\\nSummary: Icelandic detectives?\\nReview: I know Iceland is a small country and police do things a bit different in Europe but c\'mon... The incompetent police work robs this show of any believability.\\n1st Detective: \\"hey we got two persons 

In [35]:
#store the data in vector_df variable

import pandas as pd
vector_df = pd.read_json("vector_data.json", orient="records", lines=True)

In [36]:
vector_df = vector_df.sample(20000, random_state=42)

In [37]:
vector_df.shape

(20000, 7)

In [38]:
sql_df.shape

AttributeError: 'str' object has no attribute 'shape'

In [23]:
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [24]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
import chromadb

# Properly connect to the persistent database
client = chromadb.PersistentClient(path="chroma_db")

# Use get_or_create to avoid errors if it already exists
collection = client.get_or_create_collection(name="imdb_reviews2")


In [39]:
vector_df = vector_df.fillna("")

In [40]:
ids = vector_df["review_id"].astype(str).tolist()
documents = vector_df["document"].tolist()

metadatas = vector_df[[
    "movie",
    "rating",
    "spoiler_tag",
    "review_date",
    "reviewer"
]].to_dict("records")

In [41]:
vector_df["document"] = vector_df["document"].fillna("")
vector_df["document"] = vector_df["document"].astype(str)

In [42]:
ids = vector_df["review_id"].astype(str).tolist()

In [43]:
batch_size = 500

for i in tqdm(range(0, len(documents), batch_size)):

    batch_docs = documents[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]
    batch_meta = metadatas[i:i+batch_size]

    try:
        embeddings = model.encode(batch_docs).tolist()

        collection.add(
            ids=batch_ids,
            embeddings=embeddings,
            documents=batch_docs,
            metadatas=batch_meta
        )

    except Exception as e:
        print(f"Error in batch {i}: ", e)

100%|██████████| 40/40 [19:50<00:00, 29.77s/it]


In [44]:
print(collection.count())

19998


In [45]:
query = "positive reviews of Star Trek"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

print(results["documents"])

[["Movie: Star Trek: Picard (2020– )\nRating: 8.0\nSummary: Very good\nReview: Yes I agree with a lot of reviewers that this isn't really Star Trek, but it is very good. The only thing I would change is for it to be less graphic in places. It's well written and the actors are great. It is 10 times better than Star Trek discovery. What I dislike about both shows Is that the sets look more like a shopping mall than star ship cabins. (Actually there's a lot I dislike about Discovery but I won't go into that here.)", 'Movie: Star Trek: Picard (2020– )\nRating: 8.0\nSummary: Star Trek for grown ups\nReview: I have read most of the reviews here, including the ones written by those who saw all of one episode. The move away from a crew based utopian "we can solve everything in 45 minutes or less" to a more personal exploration of the effects of age, retirement, zeal and a sense of righteousness was welcome, from my point of view. It is a 21st Century Star Trek exploring inner space, not crashi

In [49]:
vector_df

,review_id,document,movie,rating,spoiler_tag,review_date,reviewer
454228,rw5345612,Movie: 蝙蝠女俠: Crisis on Infinite Earths: Part T...,蝙蝠女俠: Crisis on Infinite Earths: Part Two (201...,3.0,0,23 December 2019,andre_carreiro
529024,rw6382057,Movie: The Croods: A New Age (2020)\nRating: 6...,The Croods: A New Age (2020),6.0,0,20 December 2020,aminlv
641459,rw0975932,Movie: Anonymous Rex (2004 TV Movie)\nRating: ...,Anonymous Rex (2004 TV Movie),6.0,0,10 December 2004,Calaboss
134088,rw2883020,Movie: The Bribe (1949)\nRating: 6.0\nSummary:...,The Bribe (1949),6.0,0,7 October 2013,jhkp
175443,rw5740182,Movie: The Roads Not Taken (2020)\nRating: 7.0...,The Roads Not Taken (2020),7.0,0,15 May 2020,ramigmalkaml
...,...,...,...,...,...,...,...
461866,rw2951945,Movie: Brotherhood of Death (1976)\nRating: 6....,Brotherhood of Death (1976),6.0,0,30 January 2014,gavin6942
596709,rw6199156,Movie: Barbarians (2020– )\nRating: 8.0\nSumma...,Barbarians (2020– ),8.0,0,23 October 2020,shoham-91068
70255,rw5311693,"Movie: 重返犯罪現場：洛杉磯: Answers (2019) Season 11, E...","重返犯罪現場：洛杉磯: Answers (2019) Season 11, Episode 11",4.0,0,10 December 2019,donschmidt-39340
357922,rw6248377,Movie: The Walking Dead: World Beyond: Shadow ...,The Walking Dead: World Beyond: Shadow Puppets...,1.0,0,9 November 2020,fruizj78


BASELINE VECTOR DB TIME -- *Only vector db retrieval & HYBRID retrieval*

In [50]:
import time

query = "positive reviews of Kill chain"

start = time.time()

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

end = time.time()

print("Vector Retrieval Time:", end - start)

print(results["documents"])

Vector Retrieval Time: 0.05285978317260742
[["Movie: Kill la Kill (2013–2014)\nRating: 9.0\nSummary: Quite nice\nReview: If you happen to like anime like Tengen Toppa Gurren Lagan or FLCL, this looks like another show for you. Quick pacing, weird(and nice) shonen parody humor and everything seems to be way over the top. Five episodes in, I am already enjoying the story, although you can clearly tell that the focus is not there.The main selling point of this show is mostly it's animation, which, as usual with TV shows it gets a bit worse after the first episode. But it still very solid. You will find yourself pausing many times just to see clearly a chaotic shot up close. All in all, i recommend Kill La Kill, especially for fans of quick paced, weird anime like the ones i mentioned before.", "Movie: Kill List (2011)\nRating: 3.0\nSummary: pretentious and boring with many quarrels\nReview: To make a film incomprehensible does not make it deep. I like David Lynch and surreal films but thi

In [51]:
results = collection.query(
    query_embeddings=query_embedding,
    n_results=5,
    where={"movie": "Kill Chain"}
)

In [52]:
start = time.time()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5,
    where={"movie": "Kill Chain"}
)

end = time.time()

print("Hybrid Retrieval Time:", end - start)

Hybrid Retrieval Time: 0.008974075317382812


## Relational DB

LOAD DATA LOCAL INFILE 'D:/Adrta/HYBRID_DB_PROJECT/sql_data_mysql.csv'
INTO TABLE reviews_metadata
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\n'
IGNORE 1 LINES;


In [11]:
import pandas as pd

# load csv
df = pd.read_csv("D:/Adrta/HYBRID_DB_PROJECT/sql_data.csv")

# remove spaces from column names
df.columns = df.columns.str.strip()

# rename column x -> review_id
df = df.rename(columns={"x": "review_id"})

# count original rows
original_rows = len(df)

# remove duplicates
df_clean = df.drop_duplicates(subset="review_id")

# counts
clean_rows = len(df_clean)
duplicates_removed = original_rows - clean_rows

# save cleaned file
df_clean.to_csv("sql_data_cleaned.csv", index=False)

# print results
print("Original rows:", original_rows)
print("Rows after removing duplicates:", clean_rows)
print("Duplicates removed:", duplicates_removed)

Original rows: 20400
Rows after removing duplicates: 19994
Duplicates removed: 406


In [13]:
import pandas as pd

df = pd.read_csv("sql_data_cleaned.csv")

# convert date format
df["review_date"] = pd.to_datetime(df["review_date"]).dt.strftime("%Y-%m-%d")

df.to_csv("sql_data_mysql.csv", index=False)

print("Dates converted successfully")

Dates converted successfully


In [14]:
df.head()

,review_id,movie,reviewer,rating,review_date,spoiler_tag,review_summary
0,rw5704507,The Master (2012),indieevan,8.0,2020-05-03,0,"Great Potential, Less Great Execution"
1,rw5704535,Battle Los Angeles (2011),dwp1948,4.0,2020-05-03,0,Mediocre Alien Invasion Movie - Even Michelle ...
2,rw5704543,Code 404 (2020– ),drszilviavas,10.0,2020-05-03,0,"Brilliant comedy, amazing actors"
3,rw5704572,Halloween III: Season of the Witch (1982),adeleysim,4.0,2020-05-03,0,Ridiculous
4,rw5704612,Star Wars: The Clone Wars: Victory and Death (...,diamondsfan,10.0,2020-05-04,0,the force is strong with this one


In [ ]:
# check for duplicates
duplicates = df[df.duplicated(subset="review_id", keep=False)]
print("Duplicate review_ids:")
print(duplicates["review_id"].unique())

Duplicate review_ids:
<ArrowStringArray>
[]
Length: 0, dtype: str


In [17]:
import mysql.connector

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="root",
    database="imdb_reviews2"
)

cursor = conn.cursor()

query = """
SELECT review_id
FROM reviews_metadata
WHERE movie LIKE '%Kill%'
"""

cursor.execute(query)

candidate_ids = [str(row[0]) for row in cursor.fetchall()]

print(len(candidate_ids))
print(candidate_ids[:5])

146
['rw1988839', 'rw1993985', 'rw2480422', 'rw2483113', 'rw2492517']


## Hybrid

In [18]:
import mysql.connector

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="root",
    database="imdb_reviews2"
)

cursor = conn.cursor()

In [19]:
def sql_filter(movie_name):

    query = """
    SELECT review_id
    FROM reviews_metadata
    WHERE movie LIKE %s
    """

    cursor.execute(query, (f"%{movie_name}%",))

    ids = [str(row[0]) for row in cursor.fetchall()]

    return ids

In [58]:
candidate_ids = sql_filter("Kill")
print(len(candidate_ids))

146


In [59]:
def hybrid_search(query, movie_name, top_k=5):

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where={"movie": movie_name}
    )

    return results

In [60]:
query = "positive reviews of Kill Chain"

results = hybrid_search(query, "Kill Chain")

print(results["documents"])

[[]]


In [62]:
sample = collection.peek()
print(sample["metadatas"][:5])

[{'rating': 3.0, 'movie': '蝙蝠女俠: Crisis on Infinite Earths: Part Two (2019) Season 1, Episode 9', 'spoiler_tag': 0, 'review_date': '23 December 2019', 'reviewer': 'andre_carreiro'}, {'rating': 6.0, 'spoiler_tag': 0, 'movie': 'The Croods: A New Age (2020)', 'review_date': '20 December 2020', 'reviewer': 'aminlv'}, {'rating': 6.0, 'reviewer': 'Calaboss', 'movie': 'Anonymous Rex (2004 TV Movie)', 'review_date': '10 December 2004', 'spoiler_tag': 0}, {'spoiler_tag': 0, 'movie': 'The Bribe (1949)', 'reviewer': 'jhkp', 'review_date': '7 October 2013', 'rating': 6.0}, {'reviewer': 'ramigmalkaml', 'spoiler_tag': 0, 'movie': 'The Roads Not Taken (2020)', 'rating': 7.0, 'review_date': '15 May 2020'}]


In [64]:
query = "star trek"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

print(results["documents"])

[['Movie: Star Trek: Discovery (2017– )\nRating: 2.0\nSummary: Dull.\nReview: 1. Needs better aliens.\n2. Photon torpedoes must be deployed.\n3. Phasers not set to stun.\n4. The Borg.\n5. Janeway time travels to assume command.\n6. More robust characters, think Seven.\n7. SFx are good.', "Movie: Star Trek: Discovery (2017– )\nRating: 1.0\nSummary: Sadly not Star Trek\nReview: Every episode was a thrill. I just couldn't wait to see what strange new objects or planets or circumstances that was about to unfold on this episode. The strange remarks of Data, the inspirational storytelling capabilities of Picard, the incredible fearlessness and leadership of Janeway, the hard and internally vulnerable fatherhood of Sisko.Fighting the Borg for survival, evading Q:s elaborate and potentially reality-threatening pranks, helping planets and civilizations come together in an effort to better all of the galaxy - basically trying to do what no one has done before.That is Star Trek.Which brings me to

In [68]:
def hybrid_search(query, movie_name, top_k=5):

    # Step 1: SQL filter
    candidate_ids = sql_filter(movie_name)

    # Step 2: embed query
    query_embedding = model.encode([query]).tolist()

    # Step 3: vector search (retrieve more results)
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=30
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    ids = results["ids"][0]

    # Step 4: filter using SQL candidate ids
    filtered = [
        (doc, meta) for doc, meta, id_ in zip(docs, metas, ids)
        if id_ in candidate_ids
    ]

    # Step 5: return top_k
    return filtered[:top_k]

In [69]:
query = "positive reviews of Kill Bill"

results = hybrid_search(query, "Kill")

for r in results:
    print(r[0])

Movie: Kill la Kill (2013–2014)
Rating: 9.0
Summary: Quite nice
Review: If you happen to like anime like Tengen Toppa Gurren Lagan or FLCL, this looks like another show for you. Quick pacing, weird(and nice) shonen parody humor and everything seems to be way over the top. Five episodes in, I am already enjoying the story, although you can clearly tell that the focus is not there.The main selling point of this show is mostly it's animation, which, as usual with TV shows it gets a bit worse after the first episode. But it still very solid. You will find yourself pausing many times just to see clearly a chaotic shot up close. All in all, i recommend Kill La Kill, especially for fans of quick paced, weird anime like the ones i mentioned before.


In [77]:
import time

start = time.time()

candidate_ids = sql_filter("Kill")

end = time.time()

print("Relational Retrieval Time:", end - start)
print("Number of candidates:", len(candidate_ids))

Relational Retrieval Time: 0.027927398681640625
Number of candidates: 146


In [78]:
start = time.time()

query_embedding = model.encode(["positive reviews of Kill Bill"]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

end = time.time()

print("Vector Retrieval Time:", end - start)

Vector Retrieval Time: 0.05327892303466797


In [79]:
start = time.time()

results = hybrid_search("positive reviews of Kill Bill", "Kill")

end = time.time()

print("Hybrid Retrieval Time:", end - start)

Hybrid Retrieval Time: 0.07527351379394531


## Testing 

In [80]:
sizes = [5000, 10000, 20000]

In [81]:
import time

def benchmark(query):

    # Relational
    start = time.time()
    candidate_ids = sql_filter("Kill")
    relational_time = time.time() - start

    # Vector
    start = time.time()
    query_embedding = model.encode([query]).tolist()
    collection.query(query_embeddings=query_embedding, n_results=5)
    vector_time = time.time() - start

    # Hybrid
    start = time.time()
    hybrid_search(query, "Kill")
    hybrid_time = time.time() - start

    return relational_time, vector_time, hybrid_time

In [84]:
queries = [
    # relational
    "rating of Kill Bill",
    "reviews with rating above 8",

    # vector
    "reviews praising acting",
    "reviews criticizing storyline",

    # hybrid
    "positive reviews of Kill Bill",
    "negative reviews of Kill Bill",
    "spoiler free reviews of Kill Bill"
]

for q in queries:

    rel, vec, hyb = benchmark(q)

    print("\nQuery:", q)
    print("Relational:", rel)
    print("Vector:", vec)
    print("Hybrid:", hyb)


Query: rating of Kill Bill
Relational: 0.03390860557556152
Vector: 0.1669299602508545
Hybrid: 0.20466208457946777

Query: reviews with rating above 8
Relational: 0.026929855346679688
Vector: 0.5205240249633789
Hybrid: 0.1495981216430664

Query: reviews praising acting
Relational: 0.02668929100036621
Vector: 0.02812671661376953
Hybrid: 0.06989359855651855

Query: reviews criticizing storyline
Relational: 0.02660369873046875
Vector: 0.031259775161743164
Hybrid: 0.06224322319030762

Query: positive reviews of Kill Bill
Relational: 0.025930166244506836
Vector: 0.030917644500732422
Hybrid: 0.046903133392333984

Query: negative reviews of Kill Bill
Relational: 0.01562356948852539
Vector: 0.03490257263183594
Hybrid: 0.027674198150634766

Query: spoiler free reviews of Kill Bill
Relational: 0.03147411346435547
Vector: 0.030462980270385742
Hybrid: 0.056292057037353516


## LLM

In [ ]:
from google import genai

client = genai.Client(api_key="your_api_key")

model = "gemini-2.5-flash"

In [86]:
classifier_prompt = f"""
You are a query routing system for a hybrid database.

Classify the user query into ONE of the following categories:

RELATIONAL → structured data queries (rating, dates, counts, filters)
VECTOR → semantic meaning queries (opinions, emotions, descriptions)
HYBRID → queries needing both structure and semantics

Return ONLY one word:
RELATIONAL
VECTOR
HYBRID

User Query:
{query}
"""

In [87]:
response = client.models.generate_content(
    model=model,
    contents=classifier_prompt
)

query_type = response.text.strip()
print(query_type)

HYBRID


In [88]:
schema = """
Table: reviews_metadata

Columns:
review_id
movie
reviewer
rating
review_date
spoiler_tag
review_summary
"""

In [101]:
query = "Rating of Star Trek"

In [105]:
sql_prompt = f"""
You are a MySQL query generator.

Database schema:

Table: reviews_metadata

Columns:
review_id
movie
reviewer
rating
review_date
spoiler_tag
review_summary

Rules:
- Return ONLY a SQL query.
- Use table name: reviews_metadata
- If filtering by movie name, use LIKE with wildcards.
- Example: if query is for movie and has 2 words like 'KILL BILL' then use [ movie LIKE '%Kill%' ]
- Do NOT use explanations.

User Query:
{query}
"""

In [106]:
response = client.models.generate_content(
    model=model,
    contents=sql_prompt
)

sql_query = response.text.strip()
print(sql_query)

```sql
SELECT rating
FROM reviews_metadata
WHERE movie LIKE '%Star Trek%';
```


In [107]:
results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

docs = results["documents"][0]
context = "\n\n".join(docs)

In [108]:
answer_prompt = f"""
You are an assistant summarizing movie reviews.

User question:
{query}

Context reviews:
{context}

Based on the reviews above, generate a concise answer.

Focus on:
- common opinions
- positive and negative points
"""

In [109]:
response = client.models.generate_content(
    model=model,
    contents=answer_prompt
)

final_answer = response.text
print(final_answer)

There is no information about "Star Trek" in the provided context reviews. The reviews discuss movies and TV shows like "Kill Bill: Vol. 1", "Kill List", "The Hunt (II)", and "Killing Eve".


## Combine

In [ ]:
from sentence_transformers import SentenceTransformer
from google import genai

# embedding model for vector DB
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# gemini client
client = genai.Client(api_key="your_api_key")

llm_model = "gemini-2.5-flash-lite"

In [168]:
def classify_query(query):

    prompt = f"""
You are a query classifier for a hybrid database system.

Classify the user query into ONE of the following categories:

RELATIONAL → structured data queries (ratings, counts, dates, filters)
VECTOR → semantic opinion queries (sentiment, descriptions, opinions)
HYBRID → queries that need both structured filtering and semantic meaning

Examples:

rating of Kill Bill → RELATIONAL
reviews with rating above 8 → RELATIONAL
reviews praising acting → VECTOR
reviews criticizing storyline → VECTOR
positive reviews of Kill Bill → HYBRID
negative reviews of Kill Bill → HYBRID

Return ONLY one word: RELATIONAL, VECTOR, or HYBRID.

User Query:
{query}
"""

    response = client.models.generate_content(
    model=llm_model,
    contents=prompt
)

    return response.text.strip()

In [170]:
def vector_search(query):

    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=5
    )

    return results["documents"][0]

In [151]:
def generate_sql(query):

    prompt = f"""
You are a MySQL query generator.

Database schema:

Table: reviews_metadata

Columns:
review_id
movie
reviewer
rating
review_date
spoiler_tag
review_summary

Rules:
- Return ONLY a SQL query
- Use table name reviews_metadata
- If filtering by movie name use LIKE '%movie_name%'
- Always include context columns such as movie and review_summary when possible
- Do not return only numeric columns
- Limit results to 3

User Query:
{query}
"""

    response = client.models.generate_content(
        model=llm_model,
        contents=prompt
    )

    sql_query = response.text.strip()

    # Remove markdown if Gemini still outputs it
    sql_query = sql_query.replace("```sql", "")
    sql_query = sql_query.replace("```", "")
    sql_query = sql_query.strip()

    return sql_query

In [174]:
# def build_context(data):

#     context_list = []

#     for item in data:

#         # relational rows (tuple)
#         if isinstance(item, tuple):

#             movie = item[0]
#             review = item[1] if len(item) > 1 else ""
#             rating = item[2] if len(item) > 2 else ""

#             context_list.append(
#                 f"Movie: {movie}\nRating: {rating}\nReview: {review}"
#             )

#         else:
#             # vector or hybrid documents
#             context_list.append(item)

#     return "\n\n".join(context_list)

def build_context(data):

    context_list = []

    for i, item in enumerate(data, 1):

        if isinstance(item, tuple):

            movie = item[0]
            review = item[1] if len(item) > 1 else ""
            rating = item[2] if len(item) > 2 else ""

            context_list.append(
                f"Review {i}\nMovie: {movie}\nRating: {rating}\nText: {review}"
            )

        else:
            context_list.append(f"Review {i}\nText: {item}")

    return "\n\n".join(context_list)

In [145]:
def relational_search(query):

    sql_query = generate_sql(query)

    print("Generated SQL:", sql_query)

    cursor.execute(sql_query)

    rows = cursor.fetchall()

    # fallback to vector if empty
    if len(rows) == 0:
        print("SQL returned no results → switching to vector search")
        return vector_search(query)

    return rows

In [125]:
def hybrid_search(query, movie_name, top_k=5):

    candidate_ids = sql_filter(movie_name)

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=30
    )

    docs = results["documents"][0]
    ids = results["ids"][0]

    filtered = [
        doc for doc, id_ in zip(docs, ids)
        if id_ in candidate_ids
    ]

    return filtered[:top_k]

In [197]:
# def generate_answer(query, retrieved_data):

#     context = build_context(retrieved_data)

#     prompt = f"""
# You are an assistant summarizing movie reviews.

# User Question:
# {query}

# Retrieved Reviews:
# {context}

# Instructions:
# - Answer based only on the reviews above.
# - Mention the movie name if possible.
# - If movie is not matching, the query movie name is not found in the retrieved reviews, say "Movie not found in retrieved reviews".
# - Summarize the opinions.
# - Keep answer short (2–3 sentences).
# """

#     response = client.models.generate_content(
#         model=llm_model,
#         contents=prompt
#     )

#     return response.text

# --------------------------------------------------------------------------------------------------------

def generate_answer(query, retrieved_data):

    context = build_context(retrieved_data)

    prompt = f"""
You are a movie review assistant.

Use ONLY the information from the retrieved reviews below to answer the question.

User Question:
{query}

Retrieved Reviews:
{context}

Instructions:
- Summarize the overall opinion of the movie.
- Mention the movie name.
- Use the reviews as evidence.
- Do NOT add information not present in the reviews.
- Keep the answer concise (2–3 sentences).
"""

    response = client.models.generate_content(
        model=llm_model,
        contents=prompt
    )

    return response.text

# -------------------------------------------------------------------------------------------

# def generate_answer(query, retrieved_data):

#     context = build_context(retrieved_data)

#     prompt = f"""
# You are a movie review analysis assistant.

# User Question:
# {query}

# Retrieved Reviews:
# {context}

# Instructions:
# 1. Answer the question using only the retrieved reviews.
# 2. Summarize the overall opinion.
# 3. Provide a short answer.
# 4. List 2–3 pieces of evidence from the reviews.
# 5. Give a confidence level: High, Medium, or Low.

# Output format:

# Answer:
# ...

# Evidence:
# ...

# Confidence:
# ...
# """

#     response = client.models.generate_content(
#         model=llm_model,
#         contents=prompt
#     )

#     return response.text

In [157]:
def run_query(query):

    query_type = classify_query(query)

    print("Query type:", query_type)

    if query_type == "RELATIONAL":
        data = relational_search(query)

    elif query_type == "VECTOR":
        data = vector_search(query)

    elif query_type == "HYBRID":
        data = hybrid_search(query, "Kill")  # later extract movie automatically

    else:
        return "Unknown query type"

    answer = generate_answer(query, data)

    return answer

In [192]:
import time

def response(query):

    start = time.time()

    result = run_query(query)

    end = time.time()

    total_time = end - start

    return {
        "query": query,
        "answer": result,
        "time_seconds": total_time
    }

In [208]:
print(response("rating of any star trek"))

Query type: RELATIONAL
Generated SQL: SELECT
  movie,
  reviewer,
  rating,
  review_summary
FROM reviews_metadata
WHERE
  movie LIKE '%Star Trek%'
LIMIT 3;


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash-lite\nPlease retry in 42.055334498s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-

In [207]:
# vector    
print(response("positive review of any star trek"))

Query type: HYBRID
{'query': 'positive review of any star trek', 'answer': 'Based on the retrieved reviews, there is a positive opinion of *Star Trek* (2009). The film is described as a thrilling and exciting ride, with excellent action sequences and a well-told story that captivates the audience.', 'time_seconds': 1.6298232078552246}


In [206]:
print(response("review summary of any star trek"))

Query type: VECTOR
{'query': 'review summary of any star trek', 'answer': 'Reviews for Star Trek: Picard offer mixed opinions. One review states that the plot is "dragged out and overly complicated," with "annoying, bland and uninteresting" characters, suggesting it should be skipped. Another review, however, describes it as "Star Trek for grown ups," appreciating its focus on personal exploration and current issues.', 'time_seconds': 1.987774133682251}


In [188]:
import time

def benchmark_total(query):

    start = time.time()

    answer = run_query(query)

    end = time.time()

    total_time = end - start

    return total_time

In [189]:
def benchmark_total_system(queries):

    results = []

    for q in queries:

        start = time.time()

        answer = run_query(q)

        end = time.time()

        total_time = end - start

        results.append(total_time)

        print("\nQuery:", q)
        print("Total Time:", round(total_time,4), "sec")

    avg_time = sum(results) / len(results)

    print("\nAverage Total Time:", round(avg_time,4), "sec")

    return avg_time

In [190]:
queries = [
    "rating of Killer Elite",
    "reviews praising acting",
    "positive reviews of Killer Elite",
]

In [191]:
benchmark_total_system(queries)

Query type: RELATIONAL
Generated SQL: SELECT movie, rating, review_summary
FROM reviews_metadata
WHERE movie LIKE '%Killer Elite%'
LIMIT 3;

Query: rating of Killer Elite
Total Time: 5.7057 sec
Query type: VECTOR

Query: reviews praising acting
Total Time: 6.1513 sec
Query type: HYBRID


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 30.234819747s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'